# Chapter 2: From a candidate to a confirmed finding

The chapter companion for Chapter 2 follows one test from the historical run through the ordinary loop. This notebook shows the same loop's outcome in the fresh run: the declared objectives and selection policy, and the confirmation measurements of all thirteen frozen regions. Follow the ordinary region-search loop, then compare its frozen finalists on fresh outcomes. The predictor remains fixed throughout.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT))
read=lambda p: json.loads((R/p).read_text())
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## The frozen finalists on fresh outcomes

The protocol fixes two search objectives: the selected region's Brier contrast against the pooled rest and its population share. It also fixes the primary policy that freezes one candidate per arm and seed. The cell prints both, counts the frozen finalists and displays each finalist's confirmation share, mean Brier loss and contrast. The regions were frozen before the confirmation sample was generated.

In [2]:
protocol=read('regional/protocol.json')
print(protocol['objectives'])
print(protocol['primary_policy'])
print('Frozen candidates:',len(read('regional/frozen_finalists.json')['finalists']))
display(pd.DataFrame(results['confirmation']).T[['share','brier','contrast']])

['largest eligible region Brier contrast versus pooled rest', 'selected region population share']
maximum discovery contrast on Pareto frontier with share >= 0.10; tie larger share then earlier ID
Frozen candidates: 13


,share,brier,contrast
twinning_74101,0.10317,0.074524,0.05968
random_74101,0.10317,0.074524,0.05968
continuation_74101,0.10317,0.074524,0.05968
reflection_74101,0.106785,0.085772,0.072514
twinning_74102,0.099985,0.074805,0.059781
random_74102,0.099985,0.074805,0.059781
continuation_74102,0.099985,0.074805,0.059781
reflection_74102,0.13352,0.071811,0.058639
twinning_74103,0.111095,0.077796,0.063892
random_74103,0.11379,0.07194,0.057479


**Read the output.** Thirteen regions were frozen: four arms at each of three seeds and the subset-PAM reference. At seeds 74101 and 74102 the twinning, random and continuation arms froze the same region, so their rows are identical (contrast 0.05968 and 0.059781). The reflection arm, which used the admitted derived feature, froze a different region at every seed, with contrasts 0.072514, 0.058639 and 0.070908. Every searched region exceeds the PAM reference contrast of 0.042629, which covers a broader 20.1% of observations. One region has confirmation share 0.099985, just under the 10% floor. The floor governs discovery selection only, so confirmation measures the frozen region without adjusting it.